# miniLLM v2.3 — pré-entraînement en deux phases (WSD)

**Phase 1 (stable)** : learning rate constant sur `data/v23/stable`, jusqu'au début de la décroissance (`--stop_at`).
**Phase 2 (decay)** : on reprend le checkpoint, on change de données (`data/v23/decay`), le learning rate décroît jusqu'à `min_lr`.

Contexte 1024 (au lieu de 512), **même architecture que la v2.2** (8 têtes, 49M) : ainsi les écarts viennent des données et de la longueur de contexte, pas d'un changement d'architecture. Pour tester d'autres têtes d'attention : `N_HEADS = 16` (têtes de dimension 32), à comparer à budget égal sur un essai court.

Si la session coupe : relance simplement la même cellule (reprise automatique).

In [ ]:
# ── 1) Réglages ──
import os, sys, glob, json, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")
cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2") + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
assert BASE, "Dossier MiniLLM_v2 introuvable"
DATA, CKPT = f"{BASE}/data/v23", f"{BASE}/checkpoints"
OUT_DIR = f"{CKPT}/pretrain_v23"
MODEL_SIZE, SEQ_LEN, N_HEADS = "49M", 1024, 0          # N_HEADS = 0 : celui du preset (8) ; 16 pour l'essai « plus de têtes »
BATCH, ACCUM = 16, 4                                   # 16 x 4 x 1024 = 65 536 tokens par itération (comme la v2.2)
LR, MIN_LR, DECAY_FRAC, WARMUP = 6e-4, 6e-5, 0.15, 300
MAX_MINUTES = 0                                        # ex. 600 pour t'arrêter proprement avant la fin d'une session
CODE_DIR = "/content/minillm_v2"
!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.3 && git -C {CODE_DIR} checkout -q v2.3 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.3 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

rep = json.load(open(f"{DATA}/mix_report.json"))
TOKENS_PER_IT = BATCH * ACCUM * SEQ_LEN
N_ITERS = round(rep["total_tokens"] / TOKENS_PER_IT)
DECAY_START = int(N_ITERS * (1 - DECAY_FRAC))
print(f"{rep['total_tokens']:,} tokens / {TOKENS_PER_IT:,} par itération = {N_ITERS:,} itérations ; la décroissance commence à l'itération {DECAY_START:,}")
stable_tokens = os.path.getsize(f"{DATA}/stable/train.bin") // 2
print(f"Le fichier stable contient {stable_tokens:,} tokens, soit {stable_tokens / TOKENS_PER_IT:,.0f} itérations (il en faut {DECAY_START:,}).")
assert stable_tokens / TOKENS_PER_IT >= DECAY_START * 0.98, "phase stable trop courte pour aller jusqu'au début de la décroissance"
N_GPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
print("GPU :", N_GPU)


In [ ]:
# ── 2) PHASE 1 : stable (reprise automatique si la session coupe) ──
common = (f"--mode pretrain --out_dir {OUT_DIR} --model_size {MODEL_SIZE} --seq_len {SEQ_LEN} --batch_size {BATCH} --grad_accum {ACCUM} "
          f"--lr {LR} --min_lr {MIN_LR} --max_iters {N_ITERS} --schedule wsd --decay_frac {DECAY_FRAC} --warmup_iters {WARMUP} "
          f"--eval_every 500 --save_every 500 --eval_iters 50 --max_minutes {MAX_MINUTES}" + (f" --n_heads {N_HEADS}" if N_HEADS else ""))
def run(extra):
    cmd = f"train.py {common} {extra}"
    if N_GPU > 1:
        !torchrun --standalone --nproc_per_node={N_GPU} {cmd}
    else:
        !python {cmd}
from checkpoint import latest_checkpoint
last = latest_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
already = bool(last) and int(__import__("re").search(r"ckpt_(\d+)", last).group(1)) >= DECAY_START
print("Dernier checkpoint :", last)
if already:
    print("Phase 1 déjà terminée (checkpoint ≥ début de la décroissance).")
else:
    run(f"--data_dir {DATA}/stable --stop_at {DECAY_START}")


In [ ]:
# ── 3) PHASE 2 : decay (nouvelles données ; le learning rate descend jusqu'à min_lr) ──
# Vérifie dans les premières lignes : « Reprise à it=... » avec it = DECAY_START, puis un lr qui commence à décroître.
run(f"--data_dir {DATA}/decay")


**Après la phase 2** : évalue le checkpoint `final.pt` avec `compare_sft.py` / `knowledge_eval.py` après un SFT (comme pour la v2.2). La `val_loss` de `val.bin` est celle de la v2.2 (comparable à 3,0431) ; elle ne mesure pas les nouvelles sources, dont les documents de validation sont dans `val_new.bin`.